<a href="https://colab.research.google.com/github/muhyassin09/yasinnn/blob/main/menuju-indonesia-emas-2045/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Name: Indonesian Municipal Fiscal Analytics Framework
## Module: 01_Data_Preprocessing

**Project Pipeline Status:**
- [x] **00_Data_Acquisition.ipynb** -> Documents the source publication and how the raw table data was extracted from BPS PDF reports.
- [ ] **01_Data_Preprocessing.ipynb** *(current)* -> Loads the raw BPS fiscal CSV and runs data-quality checks (shape, dtypes, missing values, duplicates).
- [ ] **02_Data_Analysis.ipynb** -> Explores distribution shape/skewness of the 8 fiscal ratios and applies a log1p transform where it helps.
- [ ] **03_Modelling.ipynb** -> Standardizes features, selects k, fits K-Means (k=4), and profiles/visualizes the resulting clusters.
- [ ] **04_Finalizing.ipynb** -> Checks the geographic pattern of clusters, saves the final labeled dataset, and writes the project summary.

---
### 🎯 Module Objective
Load the raw 2023 BPS fiscal indicator dataset for Indonesia's 508 regencies/cities and verify it is clean (correct shape, no missing values, no duplicate regions) before any transformation happens downstream.

### 📥 Data Ingestion
* **Source File:** `data/raw/fiscal_indicators_indonesia_2023.csv`
* **Current Shape:** 508 rows × 10 columns (8 fiscal ratio indicators + `kabupaten_kota` + `provinsi`)





### 🛠️ Environment Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

print('Setup complete.')
print('pandas:', pd.__version__)
print('numpy:', np.__version__)


Setup complete.
pandas: 2.2.3
numpy: 2.1.3


## 2. Load the data

In [5]:
!git clone https://github.com/muhyassin09/yasinnn/

Cloning into 'yasinnn'...
remote: Enumerating objects: 635, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 635 (delta 37), reused 9 (delta 9), pack-reused 583 (from 1)
Receiving objects: 100% (635/635), 19.28 MiB | 17.08 MiB/s, done.
Resolving deltas: 100% (273/273), done.


In [6]:
df = pd.read_csv('/content/yasinnn/menuju-indonesia-emas-2045/data/raw/fiscal_indicators_indonesia_2023.csv', sep=';')
df.head()

,provinsi,kabupaten_kota,derajat_desentralisasi_2023,rasio_kemandirian_2023,tingkat_penyerapan_pendapatan,tingkat_penyerapan_belanja,rasio_pajak,rasio_efektivitas_pad,rasio_efektivitas_pajak,rasio_belanja_thd_pendapatan
0,Aceh,Kab. Aceh Barat,11.37,12.97,105.63,101.55,1.95,114.28,102.13,100.27
1,Aceh,Kab. Aceh Barat Daya,12.65,14.78,102.79,92.86,1.00,116.05,100.38,100.80
2,Aceh,Kab. Aceh Besar,8.59,9.56,98.88,97.43,5.53,95.99,88.80,99.09
3,Aceh,Kab. Aceh Jaya,8.39,9.25,103.83,100.24,0.80,108.48,104.93,98.53
4,Aceh,Kab. Aceh Selatan,11.95,13.77,102.20,98.92,0.95,110.79,117.74,102.19


The dataset (`fiscal_indicators_indonesia_2023.csv`) contains fiscal ratios and performance indicators for all 508 regencies/cities:

- `derajat_desentralisasi_2023` — **Decentralization Degree**: own-source revenue (PAD) as a share of total revenue. Higher = less dependent on central transfers.

- `rasio_kemandirian_2023` — **Fiscal Independence Ratio**: own-source revenue (PAD) relative to central transfers received. Higher = more self-reliant.

- `tingkat_penyerapan_pendapatan` — **Revenue Realization Rate**: actual revenue collected relative to the budget target. Higher = better at meeting revenue targets.

- `tingkat_penyerapan_belanja` — **Budget Spending Rate**: actual expenditure relative to the planned budget. Higher = more effective at utilizing allocated funds.

- `rasio_pajak` — **Local Tax Ratio**: local tax revenues as a share of total revenue. Higher = stronger capability to mobilize local taxation.

- `rasio_efektivitas_pad` — **PAD Effectiveness Ratio**: actual own-source revenue compared against original targets. Higher = more successful at exceeding internal goals.

- `rasio_efektivitas_pajak` — **Tax Effectiveness Ratio**: actual local tax collection compared against projected targets. Higher = greater efficiency in tax administration.

- `rasio_belanja_thd_pendapatan` — **Spending-to-Revenue Ratio**: total expenditure relative to total revenue. Higher = more aggressive spending, indicating deficits if above 1.0.

## 3. Initial data quality check

Before doing anything else, we check the basics: are there missing values, duplicate rows, or unexpected data types? This is a quick sanity pass — catching problems here saves confusion later.

In [7]:
print("Shape:", df.shape)
print()

print("Data types:")
print(df.dtypes)
print()

print("Missing values per column:")
print(df.isna().sum())
print()

print("Duplicate region names:", df['kabupaten_kota'].duplicated().sum())
print("Number of provinces:", df['provinsi'].nunique())

Shape: (508, 10)

Data types:
provinsi                          object
kabupaten_kota                    object
derajat_desentralisasi_2023      float64
rasio_kemandirian_2023           float64
tingkat_penyerapan_pendapatan    float64
tingkat_penyerapan_belanja       float64
rasio_pajak                      float64
rasio_efektivitas_pad            float64
rasio_efektivitas_pajak          float64
rasio_belanja_thd_pendapatan     float64
dtype: object

Missing values per column:
provinsi                         0
kabupaten_kota                   0
derajat_desentralisasi_2023      0
rasio_kemandirian_2023           0
tingkat_penyerapan_pendapatan    0
tingkat_penyerapan_belanja       0
rasio_pajak                      0
rasio_efektivitas_pad            0
rasio_efektivitas_pajak          0
rasio_belanja_thd_pendapatan     0
dtype: int64

Duplicate region names: 0
Number of provinces: 37


**Result:** 508 rows (matching BPS's official count of regencies/cities), no missing values, no duplicate region names, and 37 provinces represented. The raw data is clean.

In [9]:
df.to_csv("checked_fiscal_indicators_indonesia_2023.csv", index=False)
print("Checkpoint saved: checked_fiscal_indicators_indonesia_2023.csv")
print("Shape so far:", df.shape)


Checkpoint saved: checked_fiscal_indicators_indonesia_2023.csv
Shape so far: (508, 10)


---
### 📤 Data Export & Handoff
* **Output File:** `checked_fiscal_indicators_indonesia_2023.csv`
* **Next Destination:** `notebooks/02_Data_Analysis.ipynb`
